# 이미지 워터마크 강건성 벤치마크 (EasyPortrait / Kaggle)

구독자별 `wm_id`를 이미지에 비가시 워터마크로 넣고, **JPEG 재압축·리사이즈·크롭·스크린샷 등 변형을 거친 뒤에도** 후보 목록에서 올바른 ID를 찾아내는지 측정합니다.

**하는 일**
1. Kaggle에서 [EasyPortrait](https://www.kaggle.com/datasets/kapitanov/easyportrait)의 **일부 이미지만** 내려받기 (전체는 수십 GB라 Colab에서 불가능)
2. 워터마크 엔진 코드를 이 노트북에 기록 (아래 `%%writefile` 셀들)
3. 화질(PSNR/SSIM)과 눈에 보이는지 육안 확인
4. 공격별 검출 성공률, z 점수, 오탐 시험, 결과 CSV 저장

**먼저 알아둘 것**
- 데이터셋 라이선스는 **CC BY-SA 4.0**이며 실제 사람 얼굴 사진입니다. 사용 조건은 Kaggle 데이터셋 페이지에서 확인하세요.
- Kaggle 다운로드 셀(3번)은 **제가 직접 실행해 보지 못했습니다**(제 환경에서 Kaggle에 접속할 수 없음). 파일 목록 형식이 예상과 다르면 셀이 원인을 출력하고 멈추도록 만들었습니다. 막히면 아래 `USE_LOCAL_FOLDER`로 본인 이미지를 직접 업로드해 쓰세요.
- 그 외 셀(워터마크 엔진, 벤치마크, 분석)은 로컬 이미지로 끝까지 실행해 확인했습니다.
- **런타임은 CPU로 충분**합니다(GPU 불필요).

## 0. 설치

In [ ]:
!pip -q install kaggle kagglehub tqdm
import sys, numpy, scipy, PIL
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| scipy", scipy.__version__, "| pillow", PIL.__version__)

## 1. 설정

In [ ]:
HANDLE = "kapitanov/easyportrait"  #@param {type:"string"}
N_IMAGES = 10  #@param {type:"integer"}
MAX_SIDE = 1024  #@param {type:"integer"}
ATTACK_SET = "quick"  #@param ["tiny", "quick", "standard", "full", "full+unsupported"]
STRENGTH = 2.5  #@param {type:"number"}
N_DECOYS = 20000  #@param {type:"integer"}
WORKERS = 2  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
# 오탐 시험에 쓸 이미지 수. 워터마크가 없는 이미지는 검출이 가장 오래 걸려서(아래 8번 참고) 적게 잡았습니다.
FP_IMAGES = 3  #@param {type:"integer"}

# 본인 이미지를 쓰려면 폴더 경로를 적으세요(예: /content/my_images). 비워두면 Kaggle에서 받습니다.
USE_LOCAL_FOLDER = ""  #@param {type:"string"}
# 목록에 개별 이미지가 없고 거대한 zip만 있을 때만 True (디스크가 충분한지 확인 후)
ALLOW_FULL_DOWNLOAD = False  #@param {type:"boolean"}
MAX_FILE_MB = 30  #@param {type:"integer"}
MAX_LIST = 2000  #@param {type:"integer"}

# tiny: 3개 공격(점검용) / quick: 11개 / standard: 23개 / full: 지원 범위 전체 / full+unsupported: 회전·반전 포함
print(dict(HANDLE=HANDLE, N_IMAGES=N_IMAGES, MAX_SIDE=MAX_SIDE, ATTACK_SET=ATTACK_SET, STRENGTH=STRENGTH, N_DECOYS=N_DECOYS, WORKERS=WORKERS))

## 2. Kaggle 인증
Colab 왼쪽 **🔑 보안 비밀(Secrets)** 에 `KAGGLE_USERNAME`, `KAGGLE_KEY`를 추가하고 이 노트북에 대한 접근을 켜세요. (Kaggle → Settings → API 에서 발급)
Secrets가 없으면 `kaggle.json` 업로드를 요청합니다. Kaggle의 인증 방식이 바뀐 경우 Kaggle 안내에 따라 환경변수를 직접 설정하세요.
`USE_LOCAL_FOLDER`를 쓰면 이 셀은 건너뛰어도 됩니다.

In [ ]:
import os, json, pathlib
if not USE_LOCAL_FOLDER:
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except ImportError:
        pass
    have_env = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    have_file = pathlib.Path("~/.kaggle/kaggle.json").expanduser().exists()
    if not (have_env or have_file):
        try:
            from google.colab import files
            print("kaggle.json 을 업로드하세요.")
            up = files.upload()
            if up:
                d = pathlib.Path("~/.kaggle").expanduser(); d.mkdir(exist_ok=True)
                (d / "kaggle.json").write_bytes(next(iter(up.values()))); (d / "kaggle.json").chmod(0o600)
        except ImportError:
            raise RuntimeError("Kaggle 자격 증명이 없습니다: KAGGLE_USERNAME/KAGGLE_KEY 환경변수 또는 ~/.kaggle/kaggle.json")
    print("자격 증명:", "환경변수" if have_env else "kaggle.json")
else:
    print("로컬 폴더를 사용하므로 Kaggle 인증은 건너뜁니다.")

## 3. 이미지 가져오기
EasyPortrait는 매우 큽니다(v1 약 26GB, v2 약 91.8GB, 배포 형태는 `images.zip`). 그래서 **전체를 받지 않고** Kaggle 파일 목록에서 이미지 `N_IMAGES`장만 무작위로 골라 개별 다운로드합니다.
목록에 개별 이미지가 없고 거대한 압축 파일만 있으면 크기를 알려 주고 멈춥니다(`ALLOW_FULL_DOWNLOAD=True`로 전체 받기 가능하지만 디스크가 충분해야 합니다).

> 참고: `kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, ...)` 는 CSV 같은 **표 형식 파일용**이라 이미지에는 맞지 않아 쓰지 않았습니다.

In [ ]:
import random, shutil, zipfile
from pathlib import Path

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
ARCH_EXT = (".zip", ".tar", ".tar.gz", ".tgz", ".7z")
OUT_DIR = Path("/content/portraits") if Path("/content").exists() else Path("portraits")


def _int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None  # "12MB" 같은 문자열이면 크기를 모르는 것으로 취급


LABEL_WORDS = ("annot", "mask", "label", "segment", "depth")  # 세그멘테이션 마스크 등 사진이 아닌 파일 경로


def pick_from_listing(files, n, seed, max_mb):
    """files: [(이름, 바이트 또는 None)] -> (고른 이미지 이름들, 진단 정보). 네트워크 없이 시험할 수 있는 순수 함수."""
    imgs = [(nm, sz) for nm, sz in files if nm.lower().endswith(IMG_EXT) and (sz is None or sz <= max_mb * 1024 * 1024)]
    n_label = sum(any(w in nm.lower() for w in LABEL_WORDS) for nm, _ in imgs)
    imgs = [(nm, sz) for nm, sz in imgs if not any(w in nm.lower() for w in LABEL_WORDS)]  # 마스크/주석 이미지는 제외
    in_images_dir = [(nm, sz) for nm, sz in imgs if "image" in nm.lower()]
    if in_images_dir:  # 'images/' 폴더가 있으면 그쪽에서만 고른다
        imgs = in_images_dir
    archives = [(nm, sz) for nm, sz in files if nm.lower().endswith(ARCH_EXT)]
    chosen = random.Random(seed).sample(imgs, min(n, len(imgs)))
    return [nm for nm, _ in chosen], {"listed": len(files), "images": len(imgs), "excluded_label_like": n_label, "archives": archives}


def check_photo(path):
    """사진처럼 보이는지 검사한다. 마스크(클래스 번호 0~8 같은 값)나 거의 검은 이미지는 벤치마크를 왜곡하므로 걸러낸다."""
    import numpy as _np
    with Image.open(path) as im:
        g = _np.asarray(im.convert("L"))
    levels = int(len(_np.unique(g)))
    mean = float(g.mean())
    problems = []
    if levels < 48:
        problems.append(f"서로 다른 밝기 값이 {levels}개뿐")
    if mean < 8:
        problems.append(f"평균 밝기 {mean:.1f}로 거의 검정")
    if mean > 247:
        problems.append(f"평균 밝기 {mean:.1f}로 거의 흰색")
    return (not problems), {"levels": levels, "mean": round(mean, 1), "problems": problems}


def list_kaggle_files(handle, max_list):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    files, token = [], None
    while len(files) < max_list:
        res = api.dataset_list_files(handle, page_token=token, page_size=200)
        for f in res.files:
            name = getattr(f, "name", None) or getattr(f, "ref", None)
            size = _int(getattr(f, "total_bytes", None)) or _int(getattr(f, "size", None))
            if name:
                files.append((name, size))
        token = getattr(res, "next_page_token", None) or getattr(res, "nextPageToken", None)
        if not token:
            break
    return files


def sample_from_zip(zip_path, n, seed, out_dir):
    """큰 zip을 전부 풀지 않고 이미지 멤버 n개만 꺼낸다."""
    out = []
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if m.lower().endswith(IMG_EXT)]
        for m in random.Random(seed).sample(members, min(n, len(members))):
            dst = Path(out_dir) / Path(m).name
            dst.write_bytes(z.read(m)); out.append(dst)
    return out


def human(b):
    return "?" if b is None else f"{b / 1e9:.1f} GB" if b > 1e9 else f"{b / 1e6:.1f} MB"


OUT_DIR.mkdir(parents=True, exist_ok=True)
if USE_LOCAL_FOLDER:
    cand = sorted(p for p in Path(USE_LOCAL_FOLDER).iterdir() if p.suffix.lower() in IMG_EXT)
    if not cand:
        raise RuntimeError(f"{USE_LOCAL_FOLDER} 에 이미지가 없습니다")
    IMAGE_PATHS = [str(p) for p in random.Random(SEED).sample(cand, min(N_IMAGES, len(cand)))]
else:
    import kagglehub
    files = list_kaggle_files(HANDLE, MAX_LIST)
    names, info = pick_from_listing(files, N_IMAGES, SEED, MAX_FILE_MB)
    from collections import Counter
    print("목록의 최상위 폴더:", Counter(n.split("/")[0] for n, _ in files).most_common(8))
    print(f"목록 {info['listed']}개 중 후보 이미지 {info['images']}개 (마스크/주석처럼 보이는 경로 {info['excluded_label_like']}개 제외, 처음 {MAX_LIST}개까지만 조회)")
    if names:
        IMAGE_PATHS = []
        for nm in names:
            src = kagglehub.dataset_download(HANDLE, path=nm)
            dst = OUT_DIR / Path(nm).name
            shutil.copy(src, dst); IMAGE_PATHS.append(str(dst))
    else:
        print("개별 이미지 파일이 목록에 없습니다. 압축 파일:", [(n, human(s)) for n, s in info["archives"]] or "없음")
        if not ALLOW_FULL_DOWNLOAD:
            raise RuntimeError("개별 이미지가 없어 중단합니다. (1) USE_LOCAL_FOLDER에 본인 이미지를 올리거나, "
                               "(2) 디스크가 충분하면 ALLOW_FULL_DOWNLOAD=True 로 다시 실행하세요.")
        need = max((s or 0) for _, s in info["archives"])
        free = shutil.disk_usage(str(OUT_DIR)).free
        if need and need > free * 0.9:
            raise RuntimeError(f"디스크 부족: 압축 파일 {human(need)} / 여유 {human(free)}")
        root = Path(kagglehub.dataset_download(HANDLE))
        zips = list(root.rglob("*.zip"))
        imgs = [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXT]
        if imgs:
            IMAGE_PATHS = [str(p) for p in random.Random(SEED).sample(imgs, min(N_IMAGES, len(imgs)))]
        elif zips:
            IMAGE_PATHS = [str(p) for p in sample_from_zip(zips[0], N_IMAGES, SEED, OUT_DIR)]
        else:
            raise RuntimeError(f"{root} 에서 이미지나 zip을 찾지 못했습니다")

from PIL import Image
kept = []
print(f"{'파일':44s} {'크기':>12s} {'평균밝기':>8s} {'밝기값 수':>8s}")
for p in IMAGE_PATHS:
    ok, st = check_photo(p)
    with Image.open(p) as im:
        sz = im.size
    print(f"{Path(p).name[:44]:44s} {str(sz):>12s} {st['mean']:8.1f} {st['levels']:8d}  {'' if ok else '<- 제외: ' + ', '.join(st['problems'])}")
    if ok:
        kept.append(p)
if len(kept) < len(IMAGE_PATHS):
    print(f"\n⚠️ {len(IMAGE_PATHS) - len(kept)}장이 사진이 아닌 것으로 보여 제외했습니다 (마스크/라벨 이미지이거나 극단적으로 어두운/밝은 이미지).")
if not kept:
    raise RuntimeError("사진으로 쓸 수 있는 이미지가 없습니다. 선택된 파일이 마스크/라벨일 가능성이 큽니다. 위 표와 '최상위 폴더' 출력을 확인하고, "
                       "USE_LOCAL_FOLDER에 실제 사진을 올리세요.")
IMAGE_PATHS = kept
print(f"\n사용할 이미지 {len(IMAGE_PATHS)}장")

## 4. 워터마크 엔진 코드 기록
아래 4개 셀은 엔진 소스를 파일로 쓰기만 합니다(실행 결과 없음). `core.py`가 핵심이며, 방식은 다음과 같습니다.
- 키에서 직교하는 패턴 129개(동기용 1 + 데이터 128)를 만들고, `wm_id`의 HMAC 코드워드로 부호를 정해 합친 뒤 이미지에 **타일처럼 반복**해 휘도에 더합니다(질감이 있는 곳은 강하게, 평평한 곳은 약하게).
- 검출은 타일 주기로 스케일을 추정하고, 접어서 평균낸 뒤, 후보 `wm_id`들의 코드워드와 대조해 **z 점수**를 냅니다. 후보 수와 가설 수만큼 다중 검정 보정한 기준을 넘어야 "검출"입니다.
- **회전, 원근 변환(화면을 카메라로 촬영), 좌우 반전은 지원하지 않습니다.**

In [ ]:
%%writefile wmark/__init__.py
from .core import Codebook, SoftResult, EngineError, z_threshold, decode_image

__all__ = ["Codebook", "SoftResult", "EngineError", "z_threshold", "decode_image"]

In [ ]:
%%writefile wmark/core.py
"""
이미지 워터마킹 엔진 (확산 스펙트럼 + 타일 반복 + 후보 목록 매칭).

구조
- 키에서 직교하는 T x T 패턴(캐리어) 129개를 만든다: 동기용 파일럿 1개 + 데이터 128개.
  패턴은 저·중간 주파수 대역(기본 0.03~0.16 cycles/px)에만 에너지를 가진다 -> JPEG/리사이즈에 살아남는 대역.
- wm_id -> HMAC으로 128비트 코드워드. 비트별로 +/- 부호를 곱한 캐리어 합이 타일 패턴 W.
- W를 이미지 전체에 반복(타일링)해 휘도(Y)에 질감 마스크를 곱해 더한다.
- 검출: 주기(=스케일)를 자기상관으로 추정 -> 표준 크기로 되돌림 -> 타일로 접어 평균(호스트 잡음 감소)
  -> 파일럿 순환 상관으로 이동량(크롭) 동기 -> 128개 소프트 상관값.
- 후보 wm_id들의 코드워드와 소프트 상관값을 내적해 z 점수로 판정한다(후보 목록은 서버 DB가 가지고 있다).

한계(의도적으로 다루지 않음): 회전, 원근 변환(카메라로 화면 촬영), 뒤집기.
"""
from __future__ import annotations

import hashlib
import hmac
import io
from dataclasses import dataclass

import numpy as np
from PIL import Image
from scipy import ndimage
from scipy import fft as sfft
from scipy.stats import norm

TILE = 128
N_BITS = 128
BAND = (0.03, 0.16)
MIN_SIDE = 192  # 표준 크기로 되돌린 뒤 이 값보다 작으면 검출 불가


class EngineError(ValueError):
    pass


@dataclass
class SoftResult:
    ok: bool
    softs: list[np.ndarray] | None  # 동기 후보(이동량)별 길이 N_BITS 소프트 상관값. 첫 번째가 가장 유력
    sync_z: float  # 파일럿 상관 피크의 z 점수(동기 품질)
    period: tuple[float, float] | None  # 추정한 (x, y) 타일 주기(px)
    shift: tuple[int, int] | None
    reason: str = ""

    @property
    def n_hypotheses(self) -> int:
        """후보 매칭 시 비교하는 (스케일, 이동) 가설 수. 다중 검정 보정에 쓴다."""
        return len(self.softs or [])


def z_threshold(n_candidates: int, fwer: float = 1e-6) -> float:
    """후보 n개 중 하나라도 우연히 넘을 확률(family-wise)이 fwer 이하가 되는 z 기준."""
    return float(norm.isf(fwer / max(1, n_candidates)))


def decode_image(data: bytes) -> Image.Image:
    try:
        img = Image.open(io.BytesIO(data))
        img.load()
    except Exception as e:  # noqa: BLE001
        raise EngineError(f"cannot decode image: {e}") from e
    return img


def _luma(img: Image.Image) -> np.ndarray:
    return np.asarray(img.convert("L"), dtype=np.float64)


class Codebook:
    # 격자 탐색에서 미세 조정까지 가져갈 스케일 후보 수. 늘리면 정답이 탈락할 확률이 줄지만 검출이 느려진다.
    scale_keep = 3
    # 스케일 격자 간격(로그). 간격 중간에 진짜 값이 있으면 그 지점의 동기가 약해지므로 촘촘할수록 안정적이지만 느리다.
    scale_step = 0.015
    # 국소 스캔 후 에너지 동기 z가 이 값 미만이면 우연 수준으로 보고 포기한다 (129캐리어 실측: 무신호 스캔 최대 6.2 / 99% 5.7 (n=188), 유신호 최소 9.7 / 5% 13.0 (n=36))
    prune_z = 7.0
    # 동기 통계: "energy"(모든 캐리어 에너지 합) 또는 "pilot"(파일럿 하나)
    sync_mode = "energy"
    # 탐색 단계(반해상도)에서 동기에 쓰는 캐리어 수(파일럿 포함). 최종 읽기는 전부 쓴다. 줄이면 빨라지지만 무신호 z 분포가 바뀌어 prune_z 재보정이 필요하다
    search_carriers = 129
    # 격자 탐색 창(타일 수), 산 오르기 창(타일 수; 큰 이미지에서도 탐색 시간이 늘지 않게 상한), 국소 스캔 점 수
    grid_tiles = 4
    refine_tiles = 6
    scan_points = 9
    # 스케일 변화 없음(항등) 시도의 동기 z가 이 값 이상이면 스케일 탐색을 건너뛴다
    quick_z = 10.0
    # 후보 매칭까지 남길 서로 다른 스케일 가설의 최대 수
    max_hyp = 3

    def __init__(self, key: bytes, tile: int = TILE, n_bits: int = N_BITS, band=BAND, pilot_weight: float = 4.0):
        if len(key) < 16:
            raise EngineError("key must be at least 16 bytes")
        self.key, self.tile, self.n_bits, self.pilot_weight = key, tile, n_bits, pilot_weight
        seed = int.from_bytes(hmac.new(key, b"carriers-v1", hashlib.sha256).digest()[:8], "big")
        rng = np.random.default_rng(seed)
        m = n_bits + 1
        fy = np.fft.fftfreq(tile)[:, None]
        fx = np.fft.rfftfreq(tile)[None, :]
        r = np.hypot(fy, fx)
        mask = (r >= band[0]) & (r <= band[1])
        spec = (rng.standard_normal((m, tile, tile // 2 + 1)) + 1j * rng.standard_normal((m, tile, tile // 2 + 1))) * mask
        pats = np.fft.irfft2(spec, s=(tile, tile)).reshape(m, -1)
        q, _ = np.linalg.qr(pats.T)  # 열이 직교 정규 -> 패턴끼리 정확히 직교
        self.pats = (q.T * tile).reshape(m, tile, tile)  # 각 패턴 RMS = 1
        self._pf = sfft.rfft2(self.pats, axes=(1, 2))  # (m, T, T/2+1) 패턴 스펙트럼
        self._pf_cache: dict = {}

    # ---- 코드워드 ----
    def codeword(self, wm_id: str) -> np.ndarray:
        """wm_id(16 hex) -> 길이 n_bits 의 +/-1 벡터. 키가 없으면 계산할 수 없다."""
        out = b""
        i = 0
        while len(out) * 8 < self.n_bits:
            out += hmac.new(self.key, b"codeword-v1|" + wm_id.encode() + bytes([i]), hashlib.sha256).digest()
            i += 1
        bits = np.unpackbits(np.frombuffer(out, dtype=np.uint8))[: self.n_bits]
        return bits.astype(np.int8) * 2 - 1

    def codeword_matrix(self, ids: list[str]) -> np.ndarray:
        return np.stack([self.codeword(i) for i in ids]).astype(np.float64) if ids else np.zeros((0, self.n_bits))

    # ---- 삽입 ----
    def _tile_pattern(self, wm_id: str) -> np.ndarray:
        w = self.pilot_weight * self.pats[0] + np.tensordot(self.codeword(wm_id).astype(np.float64), self.pats[1:], axes=1)
        return w / np.sqrt(np.mean(w**2))  # RMS = 1

    @staticmethod
    def _mask(y: np.ndarray) -> np.ndarray:
        """질감이 있는 곳은 강하게, 평평한 곳(하늘 등)은 약하게."""
        mu = ndimage.uniform_filter(y, 7)
        var = np.maximum(ndimage.uniform_filter(y * y, 7) - mu * mu, 0)
        m = np.clip(np.sqrt(var) / 6.0, 0.3, 2.0)
        return ndimage.gaussian_filter(m, 3)

    def embed(self, img: Image.Image, wm_id: str, strength: float = 2.5) -> Image.Image:
        if min(img.size) < MIN_SIDE:
            raise EngineError(f"image too small: need at least {MIN_SIDE}px on each side")
        mode = img.mode
        if mode in ("L", "LA"):
            base = img.convert("L")
            y = np.asarray(base, dtype=np.float64)
            ycc = None
        else:
            ycc = img.convert("RGB").convert("YCbCr")
            y = np.asarray(ycc.getchannel("Y"), dtype=np.float64)
        h, w = y.shape
        t = self.tile
        tiled = np.tile(self._tile_pattern(wm_id), (h // t + 1, w // t + 1))[:h, :w]
        y2 = np.clip(np.rint(y + strength * self._mask(y) * tiled), 0, 255).astype(np.uint8)
        if ycc is None:
            return Image.fromarray(y2, "L")
        ch = list(ycc.split())
        ch[0] = Image.fromarray(y2, "L")
        return Image.merge("YCbCr", ch).convert("RGB")

    # ---- 검출 ----
    @staticmethod
    def _peaks(profile: np.ndarray, lo: int, hi: int, k: int = 3) -> list[float]:
        """프로파일에서 상위 k개 피크 위치(포물선 보간, 서브픽셀)."""
        hi = min(hi, len(profile) - 2)
        if hi <= lo + 2:
            return []
        seg = profile[lo:hi].copy()
        out: list[float] = []
        for _ in range(k):
            i = int(np.argmax(seg))
            if not np.isfinite(seg[i]):
                break
            j = lo + i
            a, b, c = profile[j - 1], profile[j], profile[j + 1]
            d = a - 2 * b + c
            off = 0.5 * (a - c) / d if d != 0 else 0.0
            out.append(j + max(-1.0, min(1.0, off)))
            seg[max(0, i - 10) : i + 11] = -np.inf
        return out

    def _estimate_periods(self, hp: np.ndarray) -> tuple[list[float], list[float]]:
        """
        스펙트럼을 백색화한 이미지의 자기상관에서 타일 주기(= 스케일)를 찾는다.
        백색화하면 호스트 이미지의 상관은 0 지연에만 남고 워터마크의 주기 피크만 선명해진다.
        """
        h, w = hp.shape
        f = sfft.rfft2(hp, s=(2 * h, 2 * w))
        pw = np.abs(f) ** 2
        r = np.hypot(np.fft.fftfreq(2 * h)[:, None], np.fft.rfftfreq(2 * w)[None, :])
        bins = np.linspace(0, 0.5, 101)
        idx = np.clip(np.digitize(r.ravel(), bins) - 1, 0, 99)
        tot = np.bincount(idx, pw.ravel(), 100)
        cnt = np.maximum(np.bincount(idx, minlength=100), 1)
        prof = ndimage.gaussian_filter1d(tot / cnt, 1.5) + 1e-9
        white = pw / prof[idx].reshape(pw.shape)
        white[(r < 0.02) | (r > 0.30)] = 0  # 0.30 초과는 보간 격자 등 가짜 주기
        ac = sfft.irfft2(white)
        px = ac[0, :w] / (w - np.arange(w))
        py = ac[:h, 0] / (h - np.arange(h))
        t = self.tile
        lo = int(0.35 * t)
        hi = int(3.0 * t)
        return self._peaks(px, lo, min(hi, int(0.7 * w))), self._peaks(py, lo, min(hi, int(0.7 * h)))

    # ---- 재표본화 / 접기 / 동기 (타일 크기를 인자로 받아 반해상도 탐색과 전해상도 읽기에 함께 쓴다) ----
    @staticmethod
    def _src(hp: np.ndarray):
        h, w = hp.shape
        return (Image.fromarray(np.ascontiguousarray(hp, dtype=np.float32), mode="F"), w, h, hp)

    @staticmethod
    def _canonical(src, px: float, py: float, tile: int, window=None):
        """
        주기가 정확히 tile px가 되도록 재표본화한다(window: 의심 이미지 좌표의 (x0,y0,x1,y1)).
        출력 크기는 정수, 입력 영역(box)은 실수로 지정해 크기 반올림으로 주기가 어긋나지 않게 한다
        (동기 점수의 봉우리가 +-0.5%로 좁아 주기 오차가 곧 검출 실패가 된다).
        scipy.ndimage.zoom(3차 스플라인)이 검출 시간의 97%를 차지해 PIL LANCZOS(약 7배 빠름, 축소 시 안티앨리어싱)로 바꿨다.
        """
        im, w, h, hp = src
        x0, y0, x1, y1 = window or (0.0, 0.0, float(w), float(h))
        if tile == TILE and window is None and abs(TILE / py - 1) < 1e-3 and abs(TILE / px - 1) < 1e-3:
            return hp  # 스케일 변화 없음
        ow, oh = int((x1 - x0) * tile // px), int((y1 - y0) * tile // py)
        if ow < 8 or oh < 8:
            return None
        box = (x0, y0, min(float(w), x0 + ow * px / tile), min(float(h), y0 + oh * py / tile))
        return np.asarray(im.resize((ow, oh), Image.LANCZOS, box=box), dtype=np.float32)

    @staticmethod
    def _fold(c: np.ndarray, t: int) -> np.ndarray | None:
        h, w = c.shape
        nh, nw = h // t, w // t
        if nh < 1 or nw < 1:
            return None
        return c[: nh * t, : nw * t].reshape(nh, t, nw, t).mean(axis=(0, 2))

    @staticmethod
    def _whitening(c: np.ndarray, t: int) -> np.ndarray:
        """호스트 이미지의 방사 평균 스펙트럼으로 만든 H^2 = 1/PSD (색 잡음 하에서의 정합 필터)."""
        f = sfft.rfft2(c)
        pw = np.abs(f) ** 2
        r = np.hypot(np.fft.fftfreq(c.shape[0])[:, None], np.fft.rfftfreq(c.shape[1])[None, :])
        bins = np.linspace(0, 0.5, 51)
        idx = np.clip(np.digitize(r.ravel(), bins) - 1, 0, 49)
        tot = np.bincount(idx, pw.ravel(), 50)
        cnt = np.maximum(np.bincount(idx, minlength=50), 1)
        prof = ndimage.gaussian_filter1d(tot / cnt, 1.0) + 1e-9
        rt = np.hypot(np.fft.fftfreq(t)[:, None], np.fft.rfftfreq(t)[None, :])
        it = np.clip(np.digitize(rt.ravel(), bins) - 1, 0, 49)
        return (1.0 / prof[it]).reshape(rt.shape)

    @staticmethod
    def _z(corr: np.ndarray) -> tuple[float, tuple[int, int]]:
        k = np.unravel_index(int(np.argmax(corr)), corr.shape)
        rest = np.delete(corr.ravel(), np.ravel_multi_index(k, corr.shape))
        return float((corr[k] - rest.mean()) / (rest.std() + 1e-12)), (int(k[0]), int(k[1]))

    def _carrier_spec(self, tile: int) -> np.ndarray:
        """해상도별 전체 캐리어 스펙트럼. 대역이 0.16 cycles/px 이하라 2배 솎아내기(::2)가 정확하다."""
        if tile == self.tile:
            return self._pf
        if tile not in self._pf_cache:
            step = self.tile // tile
            self._pf_cache[tile] = sfft.rfft2(self.pats[:, ::step, ::step], axes=(1, 2)).astype(np.complex64)
        return self._pf_cache[tile]

    def _corr_maps(self, folded: np.ndarray, h2: np.ndarray, tile: int, k: int | None = None) -> np.ndarray:
        """캐리어(파일럿 + 데이터)의 순환 상관 맵 (k, tile, tile). k가 없으면 전부."""
        spec = sfft.rfft2(folded)[None] * np.conj(self._carrier_spec(tile)[:k]) * h2[None]
        return sfft.irfft2(spec, s=(tile, tile), axes=(1, 2))

    def _sync_map(self, folded: np.ndarray, h2: np.ndarray, tile: int, k: int | None = None) -> np.ndarray:
        """
        이동량별 동기 통계 맵.
        energy(기본): 모든 캐리어 상관의 제곱합. 비트 부호를 몰라도 되고 파일럿(전체 에너지의 약 11%)만 쓸 때보다
        약 9배 많은 신호 에너지를 쓰므로, 타일 수가 적은 작은 이미지에서도 동기가 훨씬 잘 잡힌다.
        pilot: 파일럿 하나의 상관(이전 방식).
        """
        if self.sync_mode == "pilot":
            return sfft.irfft2(sfft.rfft2(folded) * np.conj(self._carrier_spec(tile)[0]) * h2, s=folded.shape)
        cm = self._corr_maps(folded, h2, tile, k)
        return np.einsum("kij,kij->ij", cm, cm)

    def _trial(self, src, px: float, py: float, tile: int, h2: np.ndarray | None = None, window=None):
        c = self._canonical(src, px, py, tile, window)
        if c is None or min(c.shape) < MIN_SIDE * tile // self.tile:
            return None
        f = self._fold(c, tile)
        if f is None:
            return None
        hh = h2 if h2 is not None else self._whitening(c, tile)
        z, sh = self._z(self._sync_map(f, hh, tile, self.search_carriers if tile < self.tile else None))
        return z, px, py, sh, f, hh

    def _scale_search(self, src, half: int) -> list[float]:
        """
        등방성 스케일 격자 탐색(반해상도). 각 스케일 s에서 중앙 창을 표준 크기로 되돌려 파일럿 동기 z를 잰다.
        최적 검출기를 그대로 추정기로 쓰므로 자기상관보다 훨씬 민감하다. 서로 떨어진 상위 스케일들을 돌려준다.
        """
        _, w, h, _ = src
        res: list[tuple[float, float]] = []
        for s in np.exp(np.arange(np.log(0.45), np.log(2.2), self.scale_step)):
            wy, wx = min(h, round(self.grid_tiles * 128 * s)), min(w, round(self.grid_tiles * 128 * s))
            if min(wy, wx) < 128 * s * 1.5:
                continue
            x0, y0 = (w - wx) / 2, (h - wy) / 2
            r = self._trial(src, 128 * s, 128 * s, half, None, (x0, y0, x0 + wx, y0 + wy))
            if r:
                res.append((r[0], float(s)))
        picks: list[float] = []
        for z, sc in sorted(res, reverse=True):
            if all(abs(np.log(sc / q)) > 0.05 for q in picks):
                picks.append(sc)
            if len(picks) == self.scale_keep:
                break
        return picks

    def _win(self, src, px: float, py: float):
        """산 오르기용 중앙 창: 큰 이미지에서 시도 비용이 이미지 크기에 비례하지 않도록 refine_tiles 타일로 제한한다."""
        _, w, h, _ = src
        wx, wy = min(float(w), self.refine_tiles * px), min(float(h), self.refine_tiles * py)
        x0, y0 = (w - wx) / 2, (h - wy) / 2
        return (x0, y0, x0 + wx, y0 + wy)

    def _refine(self, src, px0: float, py0: float, half: int):
        """한 시작점 주위를 촘촘히 훑고(봉우리 폭 +-0.5~0.75%보다 세밀) 이방성 미세 조정까지 한다(반해상도, 중앙 창)."""
        scan, h2 = None, None
        for d in np.linspace(-0.01, 0.01, self.scan_points):
            px, py = px0 * (1 + d), py0 * (1 + d)
            r = self._trial(src, px, py, half, h2, self._win(src, px, py))
            if r:
                h2 = r[5] if h2 is None else h2
                if scan is None or r[0] > scan[0]:
                    scan = r
        if scan is None or scan[0] < self.prune_z:
            return None  # 우연 수준(잡음 최댓값)이면 더 파지 않는다
        best = (scan[0], scan[1], scan[2])
        for step in (0.15, 0.05):
            improved, rounds = True, 0
            while improved and rounds < 3:
                improved, rounds = False, rounds + 1
                for dx, dy in ((step, 0), (-step, 0), (0, step), (0, -step)):
                    qx, qy = best[1] + dx, best[2] + dy
                    r = self._trial(src, qx, qy, half, h2, self._win(src, qx, qy))
                    if r and r[0] > best[0]:
                        best, improved = (r[0], r[1], r[2]), True
        return best

    def _polish(self, src, r):
        """반해상도에서 찾은 주기는 전해상도의 최적점에서 조금 벗어날 수 있어 동기 z를 높이는 방향으로 짧게 다듬는다."""
        best = r
        for step in (0.1, 0.04):
            improved, rounds = True, 0
            while improved and rounds < 2:
                improved, rounds = False, rounds + 1
                for dx, dy in ((step, 0), (-step, 0), (0, step), (0, -step)):
                    c = self._trial(src, best[1] + dx, best[2] + dy, self.tile, best[5])
                    if c and c[0] > best[0]:
                        best, improved = c, True
        return best

    def extract_soft(self, img: Image.Image, n_shifts: int = 3) -> SoftResult:
        y = _luma(img)
        h, w = y.shape
        if min(h, w) < MIN_SIDE // 2:
            return SoftResult(False, None, 0.0, None, None, "image too small")
        hp = (y - ndimage.gaussian_filter(y, 6)).astype(np.float32)
        src = self._src(hp)
        t = float(self.tile)
        half = self.tile // 2

        # 스케일 가설을 한 개로 확정하지 않는다: 동기 z만으로 고르면 우연히 높은 가짜 가설에 속을 수 있다.
        # 여러 가설을 남기고 후보 매칭 z(훨씬 판별력이 높음)로 최종 선택한다.
        hyps: list[tuple[float, float, float, np.ndarray, np.ndarray]] = []  # (z, px, py, folded, h2)
        ident = self._trial(src, t, t, self.tile)
        if ident:
            hyps.append((ident[0], t, t, ident[4], ident[5]))
        if not (ident and ident[0] >= self.quick_z):
            grid = [(sc * t, sc * t) for sc in self._scale_search(src, half)]
            xs, ys = self._estimate_periods(hp)
            starts = grid + sorted({(float(a), float(b)) for a in xs[:2] for b in ys[:2]})
            uniq: list[tuple[float, float]] = []
            for px, py in starts:  # 가까운 시작점(0.6% 이내)은 하나로 합친다
                if all(abs(px / a_ - 1) > 0.006 or abs(py / b_ - 1) > 0.006 for a_, b_ in uniq):
                    uniq.append((px, py))
            found = [r for r in (self._refine(src, px, py, half) for px, py in uniq) if r]
            found.sort(key=lambda r: -r[0])
            kept: list[tuple[float, float, float]] = []
            for z, px, py in found:  # 서로 다른(0.3% 초과) 상위 가설만 남긴다
                if all(abs(px / a_ - 1) > 0.003 or abs(py / b_ - 1) > 0.003 for _, a_, b_ in kept):
                    kept.append((z, px, py))
                if len(kept) == self.max_hyp:
                    break
            for rank, (_, px, py) in enumerate(kept):  # 최종 읽기는 전해상도로
                r = self._trial(src, px, py, self.tile)
                if r and rank == 0:
                    r = self._polish(src, r)  # 최상위 가설만 전해상도에서 주기를 짧게 다듬는다
                if r:
                    hyps.append((r[0], r[1], r[2], r[4], r[5]))
        if not hyps:
            return SoftResult(False, None, 0.0, None, None, "no usable scale")

        softs: list[np.ndarray] = []
        top_z, top_period, top_shift = -1.0, None, None
        for z, px, py, folded, h2 in hyps:
            cmat = self._corr_maps(folded, h2, self.tile)
            pc = np.einsum("kij,kij->ij", cmat, cmat) if self.sync_mode == "energy" else cmat[0]
            picks: list[tuple[int, int]] = []
            for o in np.argsort(-pc.ravel())[:400]:
                k = np.unravel_index(int(o), pc.shape)
                if all(min(abs(k[0] - a), self.tile - abs(k[0] - a)) > 1 or min(abs(k[1] - b), self.tile - abs(k[1] - b)) > 1 for a, b in picks):
                    picks.append((int(k[0]), int(k[1])))
                if len(picks) == n_shifts:
                    break
            softs += [cmat[1:, a, b].copy() for a, b in picks]
            if z > top_z:
                top_z, top_period, top_shift = z, (px, py), picks[0]
        return SoftResult(True, softs, top_z, top_period, top_shift, "")

    # ---- 매칭 ----
    def match(self, softs: list[np.ndarray], ids: list[str], top: int = 3, chunk: int = 50_000) -> list[tuple[str, float]]:
        """후보 wm_id들의 z 점수(내림차순 상위 top). 동기 후보(이동량) 중 가장 높은 점수를 쓴다."""
        if not ids or not softs:
            return []
        best: dict[str, float] = {}
        for start in range(0, len(ids), chunk):
            part = ids[start : start + chunk]
            b = self.codeword_matrix(part)
            zmax = np.full(len(part), -np.inf)
            for soft in softs:
                z = (b @ soft) / (np.sqrt(np.sum(soft**2)) + 1e-12)
                zmax = np.maximum(zmax, z)
            for i in np.argsort(-zmax)[:top]:
                best[part[i]] = float(zmax[i])
        return sorted(best.items(), key=lambda kv: -kv[1])[:top]

In [ ]:
%%writefile wmark/attacks.py
"""벤치마크용 공격/변환 모음. 각 함수는 PIL 이미지를 받아 변환된 PIL 이미지를 돌려준다(적용 불가면 None)."""
from __future__ import annotations

import io
import random
from typing import Callable

import numpy as np
from PIL import Image, ImageFilter, ImageOps

Attack = Callable[[Image.Image], "Image.Image | None"]


def _jpeg(q: int) -> Attack:
    def f(im: Image.Image):
        b = io.BytesIO()
        im.convert("RGB").save(b, "JPEG", quality=q)
        return Image.open(io.BytesIO(b.getvalue())).convert("RGB")
    return f


def _webp(q: int) -> Attack:
    def f(im: Image.Image):
        b = io.BytesIO()
        im.convert("RGB").save(b, "WEBP", quality=q)
        return Image.open(io.BytesIO(b.getvalue())).convert("RGB")
    return f


def _resize(s: float, resample=Image.LANCZOS) -> Attack:
    def f(im: Image.Image):
        w, h = im.size
        return im.resize((max(8, round(w * s)), max(8, round(h * s))), resample)
    return f


def _crop(frac: float, seed: int | None = None) -> Attack:
    def f(im: Image.Image):
        w, h = im.size
        cw, ch = round(w * frac), round(h * frac)
        if min(cw, ch) < 200:
            return None  # 최소 크기 미만이면 의미 있는 시험이 아님
        if seed is None:
            x0, y0 = (w - cw) // 2, (h - ch) // 2
        else:
            r = random.Random(seed)
            x0, y0 = r.randint(0, w - cw), r.randint(0, h - ch)
        return im.crop((x0, y0, x0 + cw, y0 + ch))
    return f


def _blur(sigma: float) -> Attack:
    return lambda im: im.filter(ImageFilter.GaussianBlur(sigma))


def _noise(sigma: float) -> Attack:
    def f(im: Image.Image):
        a = np.asarray(im.convert("RGB"), dtype=np.float64)
        rng = np.random.default_rng(1234)
        return Image.fromarray(np.clip(a + rng.normal(0, sigma, a.shape), 0, 255).astype(np.uint8))
    return f


def _brightness(k: float) -> Attack:
    return lambda im: Image.fromarray(np.clip(np.asarray(im.convert("RGB"), float) * k, 0, 255).astype(np.uint8))


def _contrast(k: float) -> Attack:
    def f(im: Image.Image):
        a = np.asarray(im.convert("RGB"), float)
        return Image.fromarray(np.clip((a - 128) * k + 128, 0, 255).astype(np.uint8))
    return f


def _gamma(g: float) -> Attack:
    def f(im: Image.Image):
        a = np.asarray(im.convert("RGB"), float) / 255
        return Image.fromarray((np.clip(a, 0, 1) ** g * 255).astype(np.uint8))
    return f


def _chain(*fs: Attack) -> Attack:
    def f(im: Image.Image):
        for g in fs:
            im = g(im)
            if im is None:
                return None
        return im
    return f


def _rotate(deg: float) -> Attack:
    return lambda im: im.rotate(deg, resample=Image.BICUBIC, expand=False)


ATTACKS: dict[str, Attack] = {
    "none": lambda im: im.convert("RGB"),
    "jpeg q95": _jpeg(95),
    "jpeg q85": _jpeg(85),
    "jpeg q75": _jpeg(75),
    "jpeg q60": _jpeg(60),
    "jpeg q40": _jpeg(40),
    "webp q80": _webp(80),
    "webp q60": _webp(60),
    "resize x0.9": _resize(0.9),
    "resize x0.75": _resize(0.75),
    "resize x0.6": _resize(0.6),
    "resize x1.5 (up)": _resize(1.5, Image.BICUBIC),
    "resize x2.0 (up)": _resize(2.0, Image.BICUBIC),
    "crop center 80%": _crop(0.8),
    "crop random 60%": _crop(0.6, seed=7),
    "crop random 50%": _crop(0.5, seed=11),
    "blur s1.0": _blur(1.0),
    "blur s1.5": _blur(1.5),
    "noise s3": _noise(3),
    "noise s6": _noise(6),
    "brightness x1.2": _brightness(1.2),
    "contrast x1.3": _contrast(1.3),
    "contrast x0.7": _contrast(0.7),
    "gamma 1.3": _gamma(1.3),
    "grayscale": lambda im: ImageOps.grayscale(im).convert("RGB"),
    "median 3x3": lambda im: im.convert("RGB").filter(ImageFilter.MedianFilter(3)),
    "sharpen": lambda im: im.convert("RGB").filter(ImageFilter.UnsharpMask(2, 150, 2)),
    "screenshot (x0.85+jpeg80)": _chain(_resize(0.85, Image.BILINEAR), _jpeg(80)),
    "messenger (x0.75+jpeg70)": _chain(_resize(0.75), _jpeg(70)),
    "combo crop70+x0.6+jpeg60": _chain(_crop(0.7, seed=3), _resize(0.6), _jpeg(60)),
    "combo x0.5+blur0.8+jpeg50": _chain(_resize(0.5), _blur(0.8), _jpeg(50)),
    # 아래는 지원하지 않는 변환(예상 실패). 한계를 숨기지 않기 위해 포함한다.
    "rotate 1deg (unsupported)": _rotate(1),
    "rotate 3deg (unsupported)": _rotate(3),
    "flip horizontal (unsupported)": lambda im: ImageOps.mirror(im.convert("RGB")),
}

In [ ]:
%%writefile colab_bench.py
"""
Colab/로컬 공용 벤치마크 러너 (병렬 실행, 중간 저장, 재개 지원).

- 이미지마다 고정 시드로 wm_id를 만들고, 삽입 -> 공격 -> 검출 -> 후보 매칭까지 한 작업(task)으로 처리한다.
- 결과는 한 줄씩 JSONL 파일에 즉시 기록한다. 런타임이 끊겨도 같은 파일로 다시 실행하면 끝난 작업은 건너뛴다.
- 성공 기준: 1순위 후보가 정답이고 z >= z_threshold(후보 수 x 가설 수)  (가짜 후보 n_decoys개를 섞는다)
"""
from __future__ import annotations

import json
import os
import random
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
from PIL import Image

from wmark import Codebook, z_threshold
from wmark.attacks import ATTACKS

KEY = bytes.fromhex("c0ffee" * 8)
KEY_OTHER = bytes.fromhex("badf00d1" * 8)

_QUICK = ["none", "jpeg q75", "jpeg q40", "webp q60", "resize x0.6", "resize x1.5 (up)", "crop random 50%",
          "blur s1.5", "noise s6", "screenshot (x0.85+jpeg80)", "combo crop70+x0.6+jpeg60"]
_STANDARD = _QUICK + ["jpeg q95", "jpeg q60", "resize x0.9", "resize x0.75", "crop center 80%", "crop random 60%",
                      "noise s3", "brightness x1.2", "contrast x0.7", "grayscale", "messenger (x0.75+jpeg70)",
                      "combo x0.5+blur0.8+jpeg50"]
_SUPPORTED = [k for k in ATTACKS if "unsupported" not in k]
PRESETS = {
    "tiny": ["none", "jpeg q75", "resize x0.6"],
    "quick": _QUICK,
    "standard": list(dict.fromkeys(_STANDARD)),
    "full": _SUPPORTED,
    "full+unsupported": list(ATTACKS),
}

_S: dict = {}


def prep(img: Image.Image, max_side: int) -> Image.Image:
    """서비스가 전달하는 해상도를 가정해 긴 변을 max_side로 줄인다(작으면 그대로)."""
    img = img.convert("RGB")
    w, h = img.size
    s = max_side / max(w, h)
    return img.resize((round(w * s), round(h * s)), Image.LANCZOS) if s < 1 else img


def make_id(seed: int, index: int) -> str:
    return f"{random.Random(seed * 100003 + index).getrandbits(64):016x}"


def _init(key: bytes, n_decoys: int, seed: int, strength: float) -> None:
    cb = Codebook(key)
    rng = random.Random(seed)
    ids = [f"{rng.getrandbits(64):016x}" for _ in range(n_decoys)]
    _S.update(cb=cb, decoys=ids, B=cb.codeword_matrix(ids), strength=strength, n_decoys=n_decoys)


def _zmax(softs, B):
    return np.max([(B @ s) / (np.linalg.norm(s) + 1e-12) for s in softs], axis=0)


def _task(job: dict) -> dict:
    cb: Codebook = _S["cb"]
    base = {"image": os.path.basename(job["path"]), "attack": job["attack"]}
    t0 = time.time()
    try:
        src = prep(Image.open(job["path"]), job["max_side"])
        marked = cb.embed(src, job["wm_id"], strength=_S["strength"])
        att = ATTACKS[job["attack"]](marked)
        if att is None:
            return {**base, "skipped": True, "reason": "too_small_for_attack"}
        r = cb.extract_soft(att)
        if not r.ok:
            return {**base, "ok": False, "z": None, "thr": None, "sync_z": None, "n_hyp": 0, "reason": r.reason, "sec": time.time() - t0}
        truth = cb.codeword(job["wm_id"]).astype(np.float64)[None, :]
        z_truth = float(_zmax(r.softs, truth)[0])
        z_decoy = float(_zmax(r.softs, _S["B"]).max())
        thr = z_threshold((_S["n_decoys"] + 1) * r.n_hypotheses)
        ok = bool(z_truth >= thr and z_truth > z_decoy)
        return {**base, "ok": ok, "z": z_truth, "best_decoy_z": z_decoy, "thr": thr, "sync_z": float(r.sync_z),
                "n_hyp": r.n_hypotheses, "size": list(att.size), "sec": time.time() - t0}
    except Exception as e:  # noqa: BLE001
        return {**base, "ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}", "sec": time.time() - t0}


def _fp_task(job: dict) -> dict:
    """오탐 시험: 후보 목록에 정답이 없는 경우. 기준 z 이상이 나오면 오탐."""
    cb: Codebook = _S["cb"]
    base = {"image": os.path.basename(job["path"]), "attack": job["attack"], "control": job["control"]}
    try:
        src = prep(Image.open(job["path"]), job["max_side"])
        if job["control"] == "other-key":
            src = Codebook(KEY_OTHER).embed(src, job["wm_id"], strength=_S["strength"])
        att = ATTACKS[job["attack"]](src)
        if att is None:
            return {**base, "skipped": True}
        r = cb.extract_soft(att)
        if not r.ok:
            return {**base, "checked": False, "reason": r.reason}
        z = float(_zmax(r.softs, _S["B"]).max())
        thr = z_threshold(_S["n_decoys"] * r.n_hypotheses)
        return {**base, "checked": True, "max_z": z, "thr": thr, "false_positive": bool(z >= thr)}
    except Exception as e:  # noqa: BLE001
        return {**base, "checked": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}


def _execute(fn, jobs, workers, init_args, out_path, key_fn, progress=None):
    done: dict = {}
    if out_path and os.path.exists(out_path):
        for line in open(out_path):
            try:
                r = json.loads(line)
                done[key_fn(r)] = r
            except Exception:  # noqa: BLE001
                pass
    todo = [j for j in jobs if key_fn(j) not in done]
    results = list(done.values())
    out = open(out_path, "a") if out_path else None
    bar = progress(total=len(todo), desc="벤치마크") if progress else None

    def record(r):
        results.append(r)
        if out:
            out.write(json.dumps(r) + "\n")
            out.flush()
        if bar:
            bar.update(1)

    if workers <= 1:
        _init(*init_args)
        for j in todo:
            record(fn(j))
    else:
        with ProcessPoolExecutor(max_workers=workers, initializer=_init, initargs=init_args) as ex:
            futs = [ex.submit(fn, j) for j in todo]
            for f in as_completed(futs):
                record(f.result())
    if out:
        out.close()
    if bar:
        bar.close()
    wanted = {key_fn(j) for j in jobs}
    return [r for r in results if key_fn(r) in wanted]


def run(paths, attacks, max_side=1024, strength=2.5, n_decoys=20000, workers=2, seed=42, out_path="results.jsonl", progress=None):
    jobs = [{"path": p, "attack": a, "max_side": max_side, "wm_id": make_id(seed, i)} for i, p in enumerate(paths) for a in attacks]
    key = lambda r: (os.path.basename(r.get("path", r.get("image", ""))), r["attack"])  # noqa: E731
    return _execute(_task, jobs, workers, (KEY, n_decoys, seed, strength), out_path, key, progress)


def run_fp(paths, attacks=("none", "jpeg q75", "resize x0.6", "crop random 50%", "screenshot (x0.85+jpeg80)"),
           max_side=1024, strength=2.5, n_decoys=20000, workers=2, seed=42, out_path="fp.jsonl", progress=None):
    jobs = [{"path": p, "attack": a, "control": c, "max_side": max_side, "wm_id": make_id(seed + 1, i)}
            for i, p in enumerate(paths) for c in ("unmarked", "other-key") for a in attacks]
    key = lambda r: (os.path.basename(r.get("path", r.get("image", ""))), r["attack"], r["control"])  # noqa: E731
    return _execute(_fp_task, jobs, workers, (KEY, n_decoys, seed, strength), out_path, key, progress)


def quality(paths, max_side=1024, strength=2.5):
    """원본 대비 화질(PSNR/SSIM). 값이 낮을수록 워터마크가 눈에 띌 가능성이 크다."""
    from skimage.metrics import structural_similarity

    cb = Codebook(KEY)
    rows = []
    for i, p in enumerate(paths):
        a = prep(Image.open(p), max_side)
        b = cb.embed(a, make_id(0, i), strength=strength)
        x, y = np.asarray(a, float), np.asarray(b, float)
        rows.append({"image": os.path.basename(p), "size": a.size,
                     "psnr_db": float(10 * np.log10(255**2 / max(np.mean((x - y) ** 2), 1e-9))),
                     "ssim": float(structural_similarity(np.asarray(a), np.asarray(b), channel_axis=2))})
    return rows

## 5. 화질 확인 (눈에 보이는가?)
PSNR/SSIM은 보조 지표일 뿐이라 **육안 확인이 더 중요**합니다. 아래에서 원본 · 워터마크 삽입본 · 차이(×확대)를 나란히 봅니다. 차이가 얼굴이나 매끈한 배경에서 줄무늬나 얼룩으로 보이면 `STRENGTH`를 낮추세요(검출 성공률도 같이 내려갑니다).

In [ ]:
import importlib, sys, os
importlib.invalidate_caches()
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import colab_bench as B
from wmark import Codebook

q = pd.DataFrame(B.quality(IMAGE_PATHS, MAX_SIDE, STRENGTH))
print(q.round(3).to_string(index=False))
print(f"\nPSNR 평균 {q.psnr_db.mean():.1f} dB (최소 {q.psnr_db.min():.1f}) | SSIM 평균 {q.ssim.mean():.4f} (최소 {q.ssim.min():.4f})")

cb = Codebook(B.KEY)
src = B.prep(Image.open(IMAGE_PATHS[0]), MAX_SIDE)
marked = cb.embed(src, B.make_id(0, 0), strength=STRENGTH)
a, b = np.asarray(src, float), np.asarray(marked, float)
diff = np.clip(np.abs(b - a) * 25, 0, 255).astype(np.uint8)
h, w = a.shape[:2]; cy, cx = h // 2, w // 2; s = min(128, h // 2, w // 2)
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for j, (title, im) in enumerate((("Original", src), ("Watermarked", marked), ("Difference x25", diff))):
    arr = np.asarray(im)
    ax[0, j].imshow(arr); ax[0, j].set_title(title); ax[0, j].axis("off")
    ax[1, j].imshow(arr[cy - s:cy + s, cx - s:cx + s], interpolation="nearest"); ax[1, j].set_title(title + " (center zoom)"); ax[1, j].axis("off")
plt.tight_layout(); plt.show()

## 6. 벤치마크 실행
이미지 × 공격마다 `삽입 → 공격 → 검출 → 후보 매칭`을 수행합니다. 결과는 `results.jsonl`에 **한 건씩 저장**되므로, 런타임이 끊겨도 같은 설정으로 다시 실행하면 끝난 작업은 건너뜁니다. 설정을 바꿨다면 `results.jsonl`을 지우고 다시 실행하세요(이전 설정의 결과가 섞입니다).

**시간 감각**: 속도 개선 후 제 환경(CPU 1코어, 768×1024 안팎 이미지)에서 검출 1건은 평균 약 1~3초였습니다(리사이즈·크롭처럼 스케일을 찾는 공격이 더 느리고, 확대 1.5배도 약 3초). 개선 전에는 같은 조건에서 평균 11~22초, 최대 80초 안팎이었습니다. Colab은 다를 수 있습니다.

In [ ]:
from tqdm.auto import tqdm
attacks = B.PRESETS[ATTACK_SET]
heavy = [a for a in attacks if any(k in a for k in ("resize", "crop", "screenshot", "messenger", "combo", "rotate", "flip"))]
n_tasks = len(IMAGE_PATHS) * len(attacks)
est = len(IMAGE_PATHS) * (len(heavy) * 3 + (len(attacks) - len(heavy)) * 0.7) / max(WORKERS, 1)
print(f"작업 {n_tasks}건 (이미지 {len(IMAGE_PATHS)} × 공격 {len(attacks)}), 대략 {est / 60:.0f}분 예상 (제 환경 측정값 기준의 거친 추정, 속도 개선 후)")

results = B.run(IMAGE_PATHS, attacks, max_side=MAX_SIDE, strength=STRENGTH, n_decoys=N_DECOYS,
                workers=WORKERS, seed=SEED, out_path="results.jsonl", progress=tqdm)
print("완료:", len(results), "건")

## 7. 결과

In [ ]:
df = pd.DataFrame(results)
if "error" in df.columns and df["error"].notna().any():
    print("⚠️ 오류가 난 작업:", int(df["error"].notna().sum()))
    print(df[df["error"].notna()][["image", "attack", "error"]].head(10).to_string(index=False))
df = df[~df.get("skipped", False).fillna(False).astype(bool)] if "skipped" in df.columns else df
df["ok"] = df["ok"].fillna(False).astype(bool)
order = {a: i for i, a in enumerate(attacks)}

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); m = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5)
    return ((c - m) / d, (c + m) / d)

g = df.groupby("attack").agg(성공=("ok", "sum"), 시도=("ok", "size"), z_중앙값=("z", "median"), z_최소=("z", "min")).reset_index()
g["성공률"] = (g["성공"] / g["시도"]).round(2)
g = g.sort_values("attack", key=lambda s: s.map(order)).reset_index(drop=True)
print(g.round(2).to_string(index=False))

sup = df[~df["attack"].str.contains("unsupported")]
lo, hi = wilson(int(sup.ok.sum()), len(sup))
print(f"\n지원 범위 공격 전체: {int(sup.ok.sum())}/{len(sup)} = {100 * sup.ok.mean():.1f}%   (95% 신뢰구간 {100 * lo:.0f}%~{100 * hi:.0f}%)")
print(f"검출 시 z 중앙값 {df[df.ok].z.median():.2f} | 판정 기준(평균) {df.thr.mean():.2f} | 작업당 평균 {df.sec.mean():.1f}초")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].barh(g["attack"][::-1], g["성공률"][::-1]); ax[0].set_xlim(0, 1.05); ax[0].set_title("Detection success rate by attack")
ax[1].hist([df[df.ok].z.dropna(), df[~df.ok].z.dropna()], bins=30, label=["success", "fail"], stacked=True)
ax[1].axvline(df.thr.mean(), color="k", ls="--", label="threshold"); ax[1].legend(); ax[1].set_title("z score of the true candidate")
plt.tight_layout(); plt.show()

In [ ]:
fail = df[~df.ok & ~df["attack"].str.contains("unsupported")]
print(f"실패 {len(fail)}건 (지원 범위 공격)")
if len(fail):
    print("\n[이미지별 실패 횟수]"); print(fail.groupby("image").size().sort_values(ascending=False).to_string())
    print("\n[공격별 실패한 이미지]")
    for a, grp in fail.groupby("attack"):
        print(f"  {a}: {', '.join(f'{r.image}(z={r.z:.1f})' if pd.notna(r.z) else f'{r.image}(동기 실패)' for r in grp.itertuples())}")

## 8. 오탐(false positive) 시험
후보 목록에 **정답이 없을 때**(워터마크가 없는 이미지, 다른 키로 삽입된 이미지) 기준 z를 넘는 가짜 일치가 나오는지 봅니다. 0건이어야 하고, "가짜 후보 중 최대 z"가 기준보다 충분히 낮아야 합니다.

**이 단계가 가장 느립니다.** 워터마크가 없는 이미지는 검출기가 "신호가 없다"고 결론 내리기 전에 스케일·이동 후보를 끝까지 조사하므로, 제 환경(CPU 1코어)에서 개선 전에는 건당 평균 약 22초(최대 81초), 개선 후에는 약 2초였습니다. 시간이 부족하면 `FP_IMAGES`를 줄이세요.

In [ ]:
fp_paths = IMAGE_PATHS[: min(len(IMAGE_PATHS), FP_IMAGES)]
print(f"오탐 시험 {len(fp_paths) * 2 * 5}건 실행 (이미지 {len(fp_paths)} × 대조군 2 × 공격 5)")
fp = pd.DataFrame(B.run_fp(fp_paths, max_side=MAX_SIDE, strength=STRENGTH, n_decoys=N_DECOYS, workers=WORKERS, seed=SEED, out_path="fp.jsonl", progress=tqdm))
chk = fp[fp.get("checked", False).fillna(False).astype(bool)] if "checked" in fp.columns else fp.iloc[0:0]
print(f"시험 {len(chk)}회 (워터마크 없음 / 다른 키 × 공격), 오탐 {int(chk.false_positive.sum()) if len(chk) else 0}회")
if len(chk):
    print(f"가짜 후보 중 최대 z = {chk.max_z.max():.2f} (판정 기준 {chk.thr.min():.2f}~{chk.thr.max():.2f})")
    print(chk.groupby("control").max_z.agg(["count", "max", "mean"]).round(2).to_string())

## 9. 결과 저장

In [ ]:
df.to_csv("watermark_results.csv", index=False); q.to_csv("watermark_quality.csv", index=False)
print("저장: watermark_results.csv, watermark_quality.csv, results.jsonl, fp.jsonl")
try:
    from google.colab import files
    files.download("watermark_results.csv"); files.download("watermark_quality.csv")
except ImportError:
    pass

## 결과를 읽는 법과 한계
- **성공 기준**: 1순위 후보가 정답이고 z가 (후보 수 × 가설 수)로 보정한 기준 이상일 때. 기준은 우연히 일치할 확률(family-wise)을 1e-6 이하로 맞춘 값입니다.
- **표본이 작습니다.** 이미지 몇 장 × 공격 몇 개의 결과는 신뢰구간이 넓습니다(위에 95% 구간을 표시). 이미지 1장 차이로 수치가 크게 움직입니다.
- 엔진 설계를 이 이미지들 일부로 조정했다면 같은 이미지에서의 성능은 낙관적입니다. 가능하면 **조정에 쓰지 않은 이미지**로 확인하세요.
- 이 벤치마크는 **회전, 원근 변환, 화면 카메라 촬영, 여러 사람이 이미지를 합치는 공모(collusion) 공격, 영상**을 측정하지 않습니다.
- 작은 크롭(원본 해상도 기준 대략 6타일, 즉 약 320×480px 이하)은 신호 면적이 부족해 검출이 어려워집니다.
- 실제 서비스 이미지(해상도, 촬영 조건, 전달 포맷)가 다르면 결과도 다릅니다. 최종 판단은 실제 서비스 이미지로 하세요.